In [0]:
%run ./_utils

### Export `deleted_ids.csv` (oxjob #784)

Writes the cumulative deleted-works ledger (`openalex.works.deleted_works`,
maintained nightly by `notebooks/end2end/TrackDeletedWorks`) to
`full/{date}/{format}/works/deleted_ids.csv` — placed inside BOTH format trees'
works directory (like `manifest.json`) so the entity is named by the path and
the quarterly `sync_to_public` copy picks it up with the rest of `works/`.
One row per deleted work: `work_id` (`https://openalex.org/W…`, same id form as
the works entity files) and `deleted_date` (the date the work disappeared from
`openalex_works`; for works ledgered by backfill sweeps this is the detection
date, not the true deletion date). Works that reappear are removed from the
ledger, so a live work never shows up here — the file can shrink between days.
Consumers apply the file as: remove these ids from any locally-held copy of
works.

Task ordering: depends on `export_works` because that export `rm -r`s the
`{format}/works/` directories before writing — this file must land after.


In [0]:
LEDGER = "openalex.works.deleted_works"

date_str = get_snapshot_date()
out_paths = [f"{S3_BASE}/{date_str}/{fmt}/works/deleted_ids.csv" for fmt in ("jsonl", "parquet")]
tmp_dir = f"{S3_BASE}/{date_str}/_temp/deleted_csv"

if spark.catalog.tableExists(LEDGER):
    df = (
        spark.table(LEDGER)
        .selectExpr("CONCAT('https://openalex.org/W', work_id) AS work_id", "deleted_date")
        .distinct()
        .coalesce(1)
        .sortWithinPartitions("deleted_date", "work_id")
    )
    df.write.mode("overwrite").option("header", True).csv(tmp_dir)
    part = [f.path for f in dbutils.fs.ls(tmp_dir) if f.name.startswith("part-") and f.name.endswith(".csv")][0]
    for out_path in out_paths:
        dbutils.fs.cp(part, out_path)
    dbutils.fs.rm(tmp_dir, recurse=True)
    count = spark.table(LEDGER).select("work_id").distinct().count()
    print(f"Wrote {count:,} deleted works to:")
else:
    for out_path in out_paths:
        dbutils.fs.put(out_path, "work_id,deleted_date\n", overwrite=True)
    print(f"{LEDGER} does not exist yet; wrote header-only files to:")

for out_path in out_paths:
    print(f"  {out_path}")

# One-time cleanup: 2026-08-15 shipped this file at the snapshot root before the
# works-directory placement landed. Harmless no-op on any other date.
try:
    dbutils.fs.rm(f"{S3_BASE}/{date_str}/deleted_ids.csv")
    print(f"Removed legacy root-level deleted_ids.csv from {date_str}")
except Exception:
    pass
